<a href="https://colab.research.google.com/github/UktamSam/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
print("Menu Detector!")

Menu Detector!


In [3]:
#=============================
# Import Libraries           =
#=============================
from google.colab import drive
import torchvision.transforms as transforms

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset
import os
import numpy as np

In [4]:
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Define Dataset Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print("Dataset_path:", DATASET_PATH)

CUSTOM_CLASS_MAPPING = {
    'pizza': 'pizza',
    'hot_dog': 'hot_dog',
    'chocolate_cake': 'dessert', # label grouping | class consolidation
    'cheesecake': 'dessert',     # label grouping | class consolidation
    'kebab': 'kebab',
    'plov': 'plov'
}

CLASSES = ['pizza', 'hot_dog', 'dessert', 'kebab', 'plov']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print("NUM_CLASSES:", NUM_CLASSES)
print("Class_to_idx:", CLASS_TO_IDX)

transform = transforms.Compose([                                                # compose - ketmaketlikni saqlaydi (pipline)
    transforms.Resize((224, 224)),                                              # resize - bir xil size qilib beradi
    transforms.ToTensor(),                                                      # suratlarni -> raqamlarga uguradi. 0.0 ~ 1.0 (float)
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # H W Channel => C H W. Channel: RGB
])                                                                              # Normalize - to'girlab beradi. pixel = (pixel - mean)/std

Dataset_path: /content/drive/MyDrive/food101_dataset
NUM_CLASSES: 5
Class_to_idx: {'pizza': 0, 'hot_dog': 1, 'dessert': 2, 'kebab': 3, 'plov': 4}


In [6]:
# =========================
# Custom Dataset Class    =
# =========================

class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):       #self - shu dataset object ichida saqlayapman
        self.images = images                                  #path masalan: ["pizza.jpg", "burger.jpg"]
        self.labels = labels                                  #class raqami, masalan: [0 - pizza, 1 - hot_dog]
        self.transform = transform                            #rasmga qilinadigan o‘zgarishlar

    def __len__(self):
        print('images_length', len(self.images))              # datasetni length bilib beradi
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]                           #Bu method bitta rasm va uning labelini olib beradi.
        print('image_path', img_path)
        label = self.labels[idx]
        print('label', label)
        try:
            image = Image.open(img_path).convert('RGB')       #Rasm ochiladi va RGB formatga o‘tkaziladi.
        except (UnidentifiedImageError, OSError):             #Agar rasm ochilmasa yoki buzilgan bo‘lsa, keyingi rasmga o‘tadi.
            print(f"Skipping broken image: {img_path}")
            return self.__getitem__((idx + 1) % len(self.images))

        if self.transform:
            image = self.transform(image)

        return image, label

In [7]:
# =========================
# Gather and Split Data   =
# =========================

all_images = []                                                     #Создается пустой список. Сюда будут складываться:

for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():   #классларни мэппинг килади
    class_path = os.path.join(DATASET_PATH, original_class)         #классни адресини саклайди
    print('class_path:', class_path)

    if not os.path.exists(class_path):                             #agar folder bo'lmasa -> warning
        print(f"Warning: {class_path} not found")
        continue

    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')):                 #folder ichidan bizga kerakli formatni olayabdi
            full_path = os.path.join(class_path, img)
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)

split = int(0.8 * len(all_images))
train_data = all_images[:split]                     #random qilyabmiz va 80% train uchun olyabmiz
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
img, lbl = dataset[0]

class_path: /content/drive/MyDrive/food101_dataset/pizza
class_path: /content/drive/MyDrive/food101_dataset/hot_dog
class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path: /content/drive/MyDrive/food101_dataset/cheesecake
class_path: /content/drive/MyDrive/food101_dataset/kebab
class_path: /content/drive/MyDrive/food101_dataset/plov
all_images: [('/content/drive/MyDrive/food101_dataset/cheesecake/images (3).jpeg', 2), ('/content/drive/MyDrive/food101_dataset/chocolate_cake/images (6).jpeg', 2), ('/content/drive/MyDrive/food101_dataset/kebab/Image_9.jpg', 3), ('/content/drive/MyDrive/food101_dataset/hot_dog/images (9).jpeg', 1), ('/content/drive/MyDrive/food101_dataset/pizza/images (2).jpeg', 0), ('/content/drive/MyDrive/food101_dataset/kebab/Image_25.jpg', 3), ('/content/drive/MyDrive/food101_dataset/pizza/images (7).jpeg', 0), ('/content/drive/MyDrive/food101_dataset/hot_dog/images.jpeg', 1), ('/content/drive/MyDrive/food101_dataset/chocolate_cake/images (5).jpeg